## 8 - GroupBy & Aggregation

GroupBy is one of Pandas' most powerful features. 
It follows the **Split → Apply → Combine** pattern:

```
Split   → divide DataFrame into groups based on a column
Apply   → run an aggregation function on each group
Combine → collect results back into a single DataFrame
```

### Basic syntax
```python
df.groupby('city')['score'].mean()             # mean score per city
df.groupby(['city','gender'])['score'].mean()  # multi-key groupby
```

### agg() - multiple aggregations at once
```python
df.groupby('city')['score'].agg(['mean','std','min','max','count'])
# or with custom naming:
df.groupby('city').agg(
    avg_score  = ('score', 'mean'),
    max_score  = ('score', 'max'),
    n_students = ('name', 'count')
)
```

### transform() - broadcast result back to original shape
```python
# Add city-level mean as a new column — same length as original df
df['city_avg'] = df.groupby('city')['score'].transform('mean')
```

### filter() - keep groups matching a condition
```python
df.groupby('city').filter(lambda g: g['score'].mean() > 70)
```


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('students_enriched.csv')

In [3]:
score_cols = ['maths','science','english','compSci']

In [7]:
# Basic groupby
print('Average maths score by gender:')
print(df.groupby('gender')['maths'].mean().round(2))

Average maths score by gender:
gender
Female    65.74
Male      63.96
Name: maths, dtype: float64


In [14]:
print('Average maths by study category:')
print(df.groupby('study_category')['maths'].mean().round(2).sort_values(ascending=False))

Average maths by study category:
study_category
High      65.51
Low       64.68
Medium    64.08
Name: maths, dtype: float64


In [8]:
# Multi-key groupby
print('Average by study hours AND gender:')
multi = df.groupby(['study_hours','gender'])['average'].mean().round(2)
print(multi.head(8))

Average by study hours AND gender:
study_hours  gender
3.1          Male      76.82
3.3          Female    67.00
3.4          Male      66.40
3.5          Female    69.55
3.8          Male      71.63
3.9          Female    68.87
4.3          Female    84.57
4.4          Female    66.23
Name: average, dtype: float64


In [15]:
# agg() - multiple stats at once

print('Subject stats by study category:')
stats = df.groupby('study_category').agg(
    n_students  = ('name', 'count'),
    avg_maths   = ('maths', 'mean'),
    avg_english = ('english', 'mean'),
    pass_rate   = ('passed', 'mean')
).round(2)

print(stats)

Subject stats by study category:
                n_students  avg_maths  avg_english  pass_rate
study_category                                               
High                    19      65.51        72.54        1.0
Low                     13      64.68        72.55        1.0
Medium                  18      64.08        70.74        1.0


In [16]:
# transform()
# Add group-level mean as a new column (same size as original df)

df['category_avg_maths'] = df.groupby('study_category')['maths'].transform('mean').round(2)
df['above_category_avg'] = df['maths'] > df['category_avg_maths']

print('Students above their study category average in maths:')
print(df[['name','study_category','maths','category_avg_maths','above_category_avg']].head(8))

Students above their study category average in maths:
     name study_category  maths  category_avg_maths  above_category_avg
0   Aarav            Low   52.9               64.68               False
1   Aanya         Medium   57.5               64.08               False
2   Aditi         Medium   78.7               64.08                True
3   Arjun         Medium   69.9               64.08                True
4  Bhavna           High   57.1               65.51               False
5  Chirag         Medium   72.7               64.08                True
6   Deepa         Medium   66.5               64.08                True
7   Dhruv         Medium   79.5               64.08                True


In [17]:
# filter()

print('Study categories with avg maths > 66:')

good_categories = df.groupby('study_category').filter(lambda g: g['maths'].mean() > 66)

print(good_categories['study_category'].unique())
print(f'Rows in good categories: {len(good_categories)}')

Study categories with avg maths > 66:
[]
Rows in good categories: 0


In [18]:
# Grade distribution by gender
print('Grade distribution by gender:')
print(df.groupby(['gender','grade']).size().unstack(fill_value=0))

Grade distribution by gender:
grade   A   B   C  D
gender              
Female  0   5  17  1
Male    1  12  14  0
